# Single Factor Analysis - Momentum Factor (mom_20d)

This notebook follows a standard single-factor research workflow and answers four questions:

1. Is the factor constructed and cleaned correctly?
2. Does the factor have cross-sectional predictive power?
3. Is the signal stable enough to study further?
4. Does a light Q5 portfolio still work after basic costs?

## Workflow

- Step 0: Prepare the sample, the industry+size-neutral factor, and forward-return labels.
- Step 1: Check whether the factor distribution is healthy.
- Step 2: Check IC, horizon comparison, and IC decay.
- Step 3: Check quantile returns.
- Step 4: Check the Q5-Q1 long-short spread.
- Step 5: Check stability and turnover.
- Step 6: Build a scorecard and compare industry+size neutralization with size-only neutralization.
- Step 7: Run a light research backtest with Q5 equal-weight and market-cap-weight portfolios.

## Reading Guide

For each chart, read it through four questions:

1. What question does this chart answer?
2. What do the axes and colors mean?
3. What would a good shape look like?
4. What conclusion or risk does the current chart show?

## Key Code Locations

- Data loading: `load_daily_panel` in `src/factor_utils.py`.
- Factor construction: `compute_momentum`.
- Preprocessing: `build_factor`, `neutralize_by_size`, `neutralize_by_industry_and_size`.
- Research checks: `compute_rank_ic`, `compute_ic_decay`, `compute_quantile_returns`, `long_short_spread`, `factor_autocorr`.
- Light backtest: `run_research_backtest` in this notebook. It is a research view, not a real execution simulator.


In [ ]:
from pathlib import Path
import sys, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.append(str(ROOT / "src"))
from factor_utils import (
    assign_quantiles,
    build_factor,
    compute_forward_returns,
    compute_ic_decay,
    compute_momentum,
    compute_quantile_returns,
    compute_rank_ic,
    factor_autocorr,
    load_daily_panel,
    long_short_spread,
    neutralize_by_size,
)

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", lambda x: f"{x:.4f}")

# Use English chart titles to avoid font fallback in notebook exports.
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({
    "figure.dpi": 120,
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.labelsize": 10,
    "figure.titlesize": 13,
    "legend.fontsize": 9,
    "axes.unicode_minus": False,
    "axes.edgecolor": "#D0D5DD",
    "grid.color": "#D9DEE7",
    "grid.linewidth": 0.8,
})

COLORS = {
    "blue": "#4E79A7",
    "teal": "#2A9D8F",
    "green": "#59A14F",
    "orange": "#F28E2B",
    "red": "#E15759",
    "purple": "#B07AA1",
    "gray": "#6B7280",
}
QUANTILE_COLORS = [COLORS["blue"], COLORS["teal"], COLORS["gray"], COLORS["orange"], COLORS["green"]]
HORIZON_COLORS = [COLORS["blue"], COLORS["orange"], COLORS["green"]]
C = sns.color_palette([COLORS[k] for k in ["blue", "teal", "green", "orange", "red", "purple"]])

# Parameters
FACTOR_WINDOW = 20
FORWARD_HORIZONS = [1, 5, 20]
IC_DECAY_MAX_HORIZON = 20
ALL_FORWARD_HORIZONS = sorted(set(FORWARD_HORIZONS + list(range(1, IC_DECAY_MAX_HORIZON + 1))))
N_QUANTILES = 5
DUCKDB_PATH = ROOT / "data" / "warehouse" / "ashare.duckdb"
START_DATE = "20240101"


In [ ]:
# ── 生成示例数据 ──
assert DUCKDB_PATH.exists(), f"DuckDB 文件不存在: {DUCKDB_PATH}"

df = load_daily_panel(DUCKDB_PATH, start_date=START_DATE)
print(
    f"DuckDB: {DUCKDB_PATH} | start_date={START_DATE}"
    f" | {len(df):,} rows x {df.ts_code.nunique():,} stocks x {df.trade_date.nunique():,} dates"
)


---
## Step 0. Factor Definition And Labels

This step defines the signal, the forward return labels, and the sample universe.
All later IC, quantile, and long-short analysis depends on this setup.

Current notebook setup:

- Factor value: `mom_20d = adj_close_t / adj_close_{t-20} - 1`.
- Forward labels: `fwd_1d` ... `fwd_20d`; the main evaluation label is `fwd_5d`.
- Main factor: industry + size neutralized residual, stored in `factor_industry_size_neutral`.
- Comparison factor: size-only neutralized residual, stored in `factor_size_neutral`.
- Universe filter: exclude suspended rows, missing adjusted close, missing or non-positive market cap, and stocks with fewer than `FACTOR_WINDOW + max(FORWARD_HORIZONS) + 1` listed trading days.

## Why this structure

- `compute_momentum`: defines the raw factor and writes it to `mom_20d`.
- `compute_forward_returns`: defines future return labels with no same-day overlap.
- `build_factor`: runs winsorize -> z-score -> chosen neutralization.
- `neutralize_by_size`: keeps the size-only comparison version.
- `assign_quantiles`: turns the main factor into cross-sectional buckets.


In [ ]:
# Run the standard single-factor pipeline: signal, labels, then cross-sectional processing.
df = compute_momentum(df, FACTOR_WINDOW, col_name="mom_20d", dropna=False)
df = compute_forward_returns(df, ALL_FORWARD_HORIZONS)

# Define the valid universe before later cross-sectional transforms.
min_listing_days = FACTOR_WINDOW + max(FORWARD_HORIZONS) + 1
longest_fwd_col = f"fwd_{max(FORWARD_HORIZONS)}d"
df["listed_trade_days"] = df.groupby("ts_code").cumcount() + 1
valid = (df["adj_close"].notna() & ~df["is_suspended"]
         & df["total_mv"].notna() & df["total_mv"].gt(0)
         & df["listed_trade_days"].ge(min_listing_days)
         & df["mom_20d"].notna() & df[longest_fwd_col].notna())
df["universe"] = valid

# Main preprocessing order: raw factor -> winsorized -> z-score -> industry+size neutralized.
df = build_factor(
    df,
    factor_col="mom_20d",
    neutralization="industry_size",
    output_col="factor_industry_size_neutral",
)

# Keep a size-only neutralized version for the scorecard comparison.
df = neutralize_by_size(df, "factor_zscore", output_col="factor_size_neutral")

# Use the industry+size neutral factor for the main grouped tests.
FACTOR_COL = "factor_industry_size_neutral"
SIZE_FACTOR_COL = "factor_size_neutral"
FWD_COL = "fwd_5d"
df = assign_quantiles(df, FACTOR_COL, N_QUANTILES)

print(f"Valid rows: {df.universe.sum():,} / {len(df):,}")
display(df[df.universe][[
    "trade_date", "ts_code", "sw_l1_name", "mom_20d", "factor_raw",
    "factor_zscore", SIZE_FACTOR_COL, FACTOR_COL, "quantile"
]].head(8))


---
## Step 1. Factor Distribution

This step checks whether the factor values themselves are usable before judging returns.
A broken distribution can distort IC, quantile returns, and long-short spreads.

## What This Chart Answers

- It checks whether values are concentrated, whether extreme outliers remain, and whether standardization and neutralization behaved normally.
- The x-axis is the factor value. The y-axis is the number of stock-date rows in each bin.
- The red dashed line is the mean. The dark line is the median.

## How To Read The Four Panels

- `Raw`: raw 20-day momentum.
- `Z-score`: daily cross-sectional standardized factor.
- `Size-Neutral`: size-only residual factor, kept as a comparison version.
- `Industry+Size`: the main factor after joint industry and size neutralization.

## Notes

- Healthy z-score and neutralized distributions should be centered near 0.
- This chart only says whether the factor values look usable. Predictive power is tested in later steps.


In [ ]:
sample = df[df.universe]
labels = ["Raw", "Z-score", "Size-Neutral", "Industry+Size-Neutral"]
cols = ["factor_raw", "factor_zscore", SIZE_FACTOR_COL, FACTOR_COL]
plot_colors = [COLORS["blue"], COLORS["orange"], COLORS["green"], COLORS["purple"]]

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, col, label, clr in zip(axes, cols, labels, plot_colors):
    vals = sample[col].dropna()
    ax.hist(vals, bins=40, color=clr, alpha=0.82, edgecolor="white", linewidth=0.5)
    ax.axvline(vals.mean(), color=COLORS["red"], linestyle="--", linewidth=1.2, label="Mean")
    ax.axvline(vals.median(), color="#111827", linestyle="-", linewidth=0.9, label="Median")
    ax.set_title(f"{label}\nmean={vals.mean():.3f}  std={vals.std():.3f}")
    ax.legend(frameon=False, fontsize=8)

fig.suptitle("Factor Distribution", fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

stats = []
for col, label in zip(cols, labels):
    v = sample[col].dropna()
    stats.append({"version": label, "count": len(v), "mean": v.mean(), "std": v.std(),
                  "skew": v.skew(), "p01": v.quantile(0.01), "p99": v.quantile(0.99)})
display(pd.DataFrame(stats).set_index("version"))


---
## Step 2. IC Analysis

IC measures whether today's factor ranking lines up with future return ranking.
This notebook uses Rank IC because single-factor research usually cares more about ordering than exact linear fit.

## What These Charts Answer

- `Rank IC - Daily`: daily rank correlation. The orange line is a 20-day rolling mean.
- `Cumulative IC`: cumulative daily IC. A steadily rising curve suggests persistent signal.
- `IC Distribution`: where daily IC values usually land.
- `Mean IC by Horizon`: compares 1-day, 5-day, and 20-day labels.
- `IC Decay`: shows how IC changes from horizon 1 to horizon 20.

## How To Read It

- Positive mean IC suggests higher factor names rank better on future returns.
- A win rate above 50% suggests the effect is not only from a few days.
- Slow IC decay means the signal may support a longer rebalance cycle.
- Fast IC decay means the rebalance horizon needs extra care.


In [ ]:
# Compute Rank IC for selected horizons.
research_sample = df[df.universe]
ic_frames = {}
for h in FORWARD_HORIZONS:
    ic_frames[h] = compute_rank_ic(research_sample, FACTOR_COL, f"fwd_{h}d")

ic_df = ic_frames[5]  # Main horizon.
ic_df["cum_ic"] = ic_df["rank_ic"].cumsum()
ic_df["rolling_20"] = ic_df["rank_ic"].rolling(20, min_periods=5).mean()

# Summary table.
ic_stats = []
for h in FORWARD_HORIZONS:
    ic = ic_frames[h]["rank_ic"].dropna()
    ic_stats.append({
        "horizon": f"{h}d", "mean_ic": ic.mean(), "std_ic": ic.std(ddof=0),
        "ic_ir": ic.mean() / ic.std(ddof=0) if ic.std(ddof=0) > 0 else np.nan,
        "win_rate": (ic > 0).mean(), "n_days": len(ic)
    })
display(pd.DataFrame(ic_stats).set_index("horizon"))

fig, axes = plt.subplots(2, 2, figsize=(16, 8))

ax = axes[0, 0]
ax.bar(ic_df["trade_date"], ic_df["rank_ic"], color=COLORS["blue"], alpha=0.55, width=1.5)
ax.plot(ic_df["trade_date"], ic_df["rolling_20"], color=COLORS["orange"], linewidth=2, label="20d rolling")
ax.axhline(0, color="gray", linestyle="--", linewidth=0.8)
ax.set_title("Rank IC - Daily")
ax.legend()

ax = axes[0, 1]
ax.plot(ic_df["trade_date"], ic_df["cum_ic"], color=COLORS["green"], linewidth=1.8)
ax.fill_between(ic_df["trade_date"], 0, ic_df["cum_ic"], color=COLORS["green"], alpha=0.16)
ax.axhline(0, color="gray", linestyle="--", linewidth=0.8)
ax.set_title("Cumulative IC")

ax = axes[1, 0]
ax.hist(ic_df["rank_ic"], bins=25, color=COLORS["blue"], alpha=0.78, edgecolor="white")
ax.axvline(0, color="gray", linestyle="--", linewidth=1)
ax.axvline(ic_df["rank_ic"].mean(), color=COLORS["orange"], linewidth=2,
           label=f'mean={ic_df["rank_ic"].mean():.4f}')
ax.set_title("IC Distribution")
ax.legend()

ax = axes[1, 1]
x = np.arange(len(FORWARD_HORIZONS))
means = [ic_frames[h]["rank_ic"].mean() for h in FORWARD_HORIZONS]
bars = ax.bar(x, means, color=HORIZON_COLORS, width=0.5)
ax.bar_label(bars, fmt="%.4f")
ax.set_xticks(x)
ax.set_xticklabels([f"{h}d" for h in FORWARD_HORIZONS])
ax.axhline(0, color="gray", linestyle="--", linewidth=0.8)
ax.set_title("Mean IC by Horizon")

fig.suptitle("IC Analysis", fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

# IC decay: IC(t, t+h) for h=1..N.
decay_df = compute_ic_decay(research_sample, FACTOR_COL, max_lag=IC_DECAY_MAX_HORIZON)
display(decay_df.set_index("horizon"))

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(decay_df["horizon"], decay_df["mean_ic"], color=COLORS["blue"], linewidth=2, marker="o")
ax.fill_between(
    decay_df["horizon"],
    decay_df["mean_ic"] - decay_df["std_ic"],
    decay_df["mean_ic"] + decay_df["std_ic"],
    color=COLORS["blue"],
    alpha=0.15,
)
ax.axhline(0, color="gray", linestyle="--", linewidth=0.8)
ax.set_title("IC Decay by Horizon")
ax.set_xlabel("Forward Horizon (days)")
ax.set_ylabel("Mean Rank IC")
plt.tight_layout()
plt.show()


---
## Step 3. 分层收益

这一步对应单因子检测里的“经济含义验证”。IC 告诉我们排序相关性，分层收益告诉我们这个排序能不能转成更直观的收益差。

做法是：每天把股票按因子值分成 5 组，然后看每组未来收益。

- Q1：因子最低的一组。
- Q5：因子最高的一组。
- 如果因子方向是正的，我们希望 Q5 的未来收益高于 Q1。

## 这组图回答什么

- 📌 它回答：因子分数越高，未来收益是否越高。
- 👀 左图 `Mean Return by Quantile`：每根柱子是一组股票的平均未来收益。
- 👀 右图 `Cumulative Return by Quantile`：每组收益随时间累积后的曲线。
- ✅ 最理想形态：Q1、Q2、Q3、Q4、Q5 从低到高大体递增，右图里 Q5 长期跑在 Q1 上方。

## 新手怎么看好坏

- ✅ 单调上升最重要：不是只要求 Q5 高，而是希望从低分到高分逐步变好。
- ✅ Q5 和 Q1 差距越清楚，说明因子把好股票和差股票分得越开。
- ⚠️ 如果中间组乱序，说明因子有方向，但排序质量不够细腻。
- ⚠️ 如果只有 Q5 很高、其他组没规律，可能只是头部股票贡献，稳定性要打折。

## 当前样例怎么读

当前样例里，高分组整体更好，说明动量因子的正方向是存在的。但评分卡里会进一步检查 Q1 到 Q5 是否严格单调；如果不严格单调，就代表这个因子的排序能力还不够完美，需要后续用真实数据、行业中性化、交易成本等继续验证。


In [ ]:
# Parameters
valid = df[df.universe & df["quantile"].notna() & df[FWD_COL].notna()]
q_summary, daily_q, q_pivot = compute_quantile_returns(valid, FWD_COL)
q_summary = q_summary.set_index("quantile")
q_cum = (1 + q_pivot.fillna(0)).cumprod() - 1

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
bars = ax.bar(q_summary.index.astype(str), q_summary["mean_return"],
              color=QUANTILE_COLORS, edgecolor="white", linewidth=0.8)
for bar, val in zip(bars, q_summary["mean_return"]):
    offset = 0.0002 if val >= 0 else -0.0002
    va = "bottom" if val >= 0 else "top"
    ax.text(bar.get_x() + bar.get_width()/2, val + offset,
            f"{val:.4%}", ha="center", va=va, fontsize=9, color="#374151")
ax.axhline(0, color=COLORS["gray"], linestyle="--", linewidth=0.8)
ax.set_title(f"Mean {FWD_COL} Return by Quantile")
ax.set_xlabel("Quantile (1=Low, 5=High)")

ax = axes[1]
for q in range(1, N_QUANTILES + 1):
    if q in q_cum.columns:
        ax.plot(q_cum.index, q_cum[q], color=QUANTILE_COLORS[q - 1],
                linewidth=2.0 if q in (1, N_QUANTILES) else 1.6,
                alpha=0.96, label=f"Q{q}")
ax.axhline(0, color=COLORS["gray"], linestyle="--", linewidth=0.8)
ax.set_title("Cumulative Return by Quantile")
ax.legend(ncol=5, fontsize=8, frameon=True)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))

fig.suptitle("Quantile Return Analysis", fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

display(q_summary)


---
## Step 4. 多空收益

这一步对应单因子检测里的“把分层结果压缩成一个最容易比较的指标”。

最常见的做法就是 `Q5 - Q1`：买最高分组，卖最低分组，观察两者之间的收益差。它不是为了马上实盘交易，而是为了快速判断“高分股相对低分股到底有没有优势”。

## 这组图回答什么

- 📌 它回答：高分组相对低分组的收益差是否稳定为正。
- 👀 左图 `Daily Spread`：每天 Q5 收益减 Q1 收益。柱子在 0 上方代表当天高分组赢了低分组。
- 👀 右图 `Cumulative Spread`：把每日 spread 累加。曲线向上代表长期高分组持续跑赢低分组。

## 新手怎么看好坏

- ✅ 日度 spread 平均值为正：说明高分组整体占优。
- ✅ 累计 spread 越平滑向上越好：说明优势不是少数几天贡献的。
- ⚠️ 如果累计曲线大起大落，说明虽然最终赚钱，但过程很不稳定。
- ⚠️ 如果分层收益看起来不错，但 Q5-Q1 很弱，说明因子方向存在，但强度不够。

## 当前样例怎么读

当前样例里，Q5-Q1 的平均 spread 和累计 spread 都是正的，所以从研究结论层面看，这个因子能把高低分股票拉开差距。但交易落地还不能直接下结论，因为这里还没有加入交易成本、冲击成本、涨跌停、停牌和融券限制。


In [ ]:
# Charts.
ls = long_short_spread(q_pivot)

# Summary.
s = ls["spread"].dropna()
cum = (1 + s).cumprod()
dd = (cum / cum.cummax() - 1)
ls_stats = pd.DataFrame({
    "metric": ["mean_spread", "volatility", "win_rate", "cum_return", "max_drawdown", "sharpe"],
    "value": [s.mean(), s.std(), (s > 0).mean(),
              ls["cum_spread"].iloc[-1], dd.min(),
              s.mean() / s.std() if s.std() > 0 else np.nan]
})
display(ls_stats.set_index("metric"))

# Summary. ?? ??
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.bar(ls["trade_date"], ls["spread"],
       color=[COLORS["green"] if v >= 0 else COLORS["red"] for v in s], alpha=0.6, width=1.5)
ax.axhline(0, color="gray", linestyle="--", linewidth=0.8)
ax.set_title(f"Q5 - Q1 Daily Spread (mean={s.mean():.4%})")

ax = axes[1]
ax.plot(ls["trade_date"], ls["cum_spread"], color=COLORS["green"], linewidth=2)
ax.fill_between(ls["trade_date"], 0, ls["cum_spread"], color=COLORS["green"], alpha=0.16)
ax.axhline(0, color="gray", linestyle="--", linewidth=0.8)
ax.set_title(f'Q5 - Q1 Cumulative Spread (final={ls["cum_spread"].iloc[-1]:.4%})')
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))

fig.suptitle("Long-Short Analysis", fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()


---
## Step 5. 稳定性检查

这一步对应单因子检测里的“不要只看均值，要看这个因子是不是经得起时间和调仓”。

一个因子可能平均有效，但如果只在前半段有效、排序每天乱跳、换手特别高，实盘价值就会明显下降。

## 这组图回答什么

- 📌 它回答：因子是不是稳定、可持续、可调仓。
- 👀 `20-Day Rolling IC`：看 IC 的短期趋势。橙线持续在 0 上方更好。
- 👀 `Factor Rank Autocorrelation`：看今天和昨天的因子排序像不像。越接近 1，说明排序越稳定。
- 👀 `Quantile Turnover`：看每个分组每天换掉多少股票。越高，代表调仓越频繁。

## 新手怎么看好坏

- ✅ 滚动 IC 长期为正：说明预测力不是只出现在某一小段时间。
- ✅ 秩自相关较高：说明股票排序不乱跳，组合更容易维护。
- ✅ 换手率不过高：说明交易成本更容易控制。
- ⚠️ 秩自相关太低，代表每天选出来的股票变化很大，策略会很难落地。
- ⚠️ 换手率太高，即使纸面收益不错，也可能被交易成本吃掉。

## 当前样例怎么读

当前样例里，排序自相关较高，说明因子排序比较稳定；换手率不是零，说明组合会发生正常调整。结合前后半段 IC 对比，如果前后表现差异不大，就可以说这个因子在样例数据里不是“一阵风式”的偶然有效。


In [ ]:
# 1) Compare IC between first and second halves.
mid = len(ic_df) // 2
half_stats = []
for label, part in [("Full sample", ic_df), ("First half", ic_df.iloc[:mid]), ("Second half", ic_df.iloc[mid:])]:
    ic = part["rank_ic"].dropna()
    half_stats.append({
        "period": label, "n": len(ic), "mean_ic": ic.mean(),
        "win_rate": (ic > 0).mean(),
        "ic_ir": ic.mean() / ic.std() if ic.std() > 0 else np.nan
    })
half_df = pd.DataFrame(half_stats)
display(half_df.set_index("period"))

# 2) Factor rank autocorrelation.
autocorr_df = factor_autocorr(df[df.universe], FACTOR_COL)

# 3) Quantile turnover.
dates = sorted(df[df.universe & df["quantile"].notna()]["trade_date"].unique())
turnover_rows = []
for d1, d2 in zip(dates[:-1], dates[1:]):
    prev = df[(df.trade_date == d1) & df["quantile"].notna()]
    curr = df[(df.trade_date == d2) & df["quantile"].notna()]
    for q in range(1, N_QUANTILES + 1):
        p_set = set(prev[prev["quantile"] == q]["ts_code"])
        c_set = set(curr[curr["quantile"] == q]["ts_code"])
        if len(c_set) > 0:
            turnover_rows.append({"trade_date": d2, "quantile": q,
                                  "turnover": 1 - len(p_set & c_set) / len(c_set)})
turnover_df = pd.DataFrame(turnover_rows)
t_summary = turnover_df.groupby("quantile")["turnover"].mean().reset_index()
t_summary.columns = ["quantile", "avg_turnover"]

# Summary.
display(pd.DataFrame({
    "metric": ["mean_autocorr", "mean_turnover"],
    "value": [autocorr_df["autocorr"].mean(), turnover_df["turnover"].mean()]
}).set_index("metric"))

# Summary. ?? ??
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))

ax = axes[0]
ax.bar(ic_df["trade_date"], ic_df["rank_ic"], color=COLORS["blue"], alpha=0.35, width=1.5)
ax.plot(ic_df["trade_date"], ic_df["rolling_20"], color=COLORS["orange"], linewidth=2)
ax.axhline(0, color="gray", linestyle="--", linewidth=0.8)
ax.set_title("20-Day Rolling IC")

ax = axes[1]
ax.plot(autocorr_df["trade_date"], autocorr_df["autocorr"], color=COLORS["green"], linewidth=1.8)
ax.axhline(0, color="gray", linestyle="--", linewidth=0.8)
ax.set_title("Factor Rank Autocorrelation")

ax = axes[2]
for q in range(1, N_QUANTILES + 1):
    part = turnover_df[turnover_df["quantile"] == q]
    ax.plot(part["trade_date"], part["turnover"], color=QUANTILE_COLORS[q - 1], linewidth=1.4, alpha=0.9, label=f"Q{q}")
ax.set_title("Quantile Turnover")
ax.legend(ncol=5, fontsize=8)

fig.suptitle("Stability Analysis", fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()


---
## Step 6. Scorecard And Research Conclusion

This step combines the evidence from IC, quantile returns, long-short spread, stability, and turnover.
The goal is not to over-read one number, but to judge whether the factor deserves more research.

## What The Scorecard Answers

- `Mean IC`: average prediction direction.
- `IC Win Rate`: share of positive IC days.
- `IC IR`: mean IC divided by IC volatility.
- `Q1 -> Q5 Monotonic`: whether quantile returns improve from low factor to high factor.
- `Q5 - Q1 Mean Spread`: average advantage of high factor names over low factor names.
- `Neutralization Compare`: whether industry+size and size-only versions point in the same direction.
- `Rank Autocorr`: factor ranking stability.
- `Avg Turnover`: rebalancing pressure.

## Reading Order

1. Check Mean IC and IC IR.
2. Check monotonicity.
3. Check Q5-Q1 spread.
4. Compare industry+size neutralization against size-only neutralization.
5. Check stability and turnover.

## Current Research Position

- The main factor now uses joint industry and size neutralization.
- The scorecard keeps the size-only version as a direct comparison.
- If monotonicity is weak, it means the signal needs more refinement, not necessarily that the direction is useless.
- A production strategy would still need fuller execution constraints, impact costs, and a longer sample.


In [ ]:
# Scorecard
ic_mean = ic_df["rank_ic"].mean()
ic_win = (ic_df["rank_ic"] > 0).mean()
ic_std = ic_df["rank_ic"].std(ddof=0)
ls_mean = ls["spread"].mean()
ls_cum = ls["cum_spread"].iloc[-1]
auto_mean = autocorr_df["autocorr"].mean()
turnover_mean = turnover_df["turnover"].mean()

comparison_rows = []
for version, factor_col in [("Size-neutral", SIZE_FACTOR_COL), ("Industry+size-neutral", FACTOR_COL)]:
    version_df = assign_quantiles(research_sample.copy(), factor_col, N_QUANTILES)
    version_q_summary, _, version_q_pivot = compute_quantile_returns(version_df, FWD_COL)
    version_ls = long_short_spread(version_q_pivot)
    version_ic = compute_rank_ic(version_df, factor_col, FWD_COL)["rank_ic"].dropna()
    comparison_rows.append({
        "version": version,
        "mean_ic": version_ic.mean(),
        "q5_q1_mean_spread": version_ls["spread"].mean(),
        "q5_q1_cum_return": version_ls["cum_spread"].iloc[-1],
        "q5_mean_return": version_q_summary.loc[
            version_q_summary["quantile"].eq(N_QUANTILES), "mean_return"
        ].iloc[0],
    })
neutralization_compare = pd.DataFrame(comparison_rows).set_index("version")
display(neutralization_compare)

mean_returns = q_summary["mean_return"]
if mean_returns.is_monotonic_increasing:
    monotonic_state = "Positive monotonic"
    monotonic_judgement = "Pass"
elif mean_returns.is_monotonic_decreasing:
    monotonic_state = "Negative monotonic"
    monotonic_judgement = "Reverse"
else:
    monotonic_state = "No"
    monotonic_judgement = "Review"

scorecard = pd.DataFrame([
    ["Main factor version", "Industry + size neutral", "-"],
    ["Mean IC", f"{ic_mean:.4f}", "Pass" if ic_mean > 0 else "Weak"],
    ["IC Win Rate", f"{ic_win:.1%}", "Pass" if ic_win > 0.5 else "Weak"],
    ["IC IR", f"{ic_mean / ic_std:.3f}" if ic_std > 0 else "nan", "-"],
    ["Q1 to Q5 monotonic", monotonic_state, monotonic_judgement],
    ["Q5 - Q1 mean spread", f"{ls_mean:.4%}", "Pass" if ls_mean > 0 else "Weak"],
    ["Long-short cumulative", f"{ls_cum:.4%}", "Pass" if ls_cum > 0 else "Weak"],
    ["Size-only Mean IC", f"{neutralization_compare.loc['Size-neutral', 'mean_ic']:.4f}", "Compare"],
    ["Size-only Q5-Q1", f"{neutralization_compare.loc['Size-neutral', 'q5_q1_mean_spread']:.4%}", "Compare"],
    ["Rank autocorr", f"{auto_mean:.3f}", "-"],
    ["Avg turnover", f"{turnover_mean:.1%}", "-"],
], columns=["Metric", "Value", "Judgement"])

def scorecard_row_style(row):
    base = "color: #111827; border-color: #E5E7EB;"
    if row["Judgement"] == "Pass":
        bg = "background-color: #D1FAE5;"
    elif row["Judgement"] == "Weak":
        bg = "background-color: #FEE2E2;"
    elif row["Judgement"] == "Reverse":
        bg = "background-color: #DBEAFE;"
    elif row["Judgement"] == "Review":
        bg = "background-color: #FEF3C7;"
    elif row["Judgement"] == "Compare":
        bg = "background-color: #E0F2FE;"
    else:
        bg = "background-color: #F8FAFC;"
    return [bg + base for _ in row]

scorecard_style = (
    scorecard.style
    .apply(scorecard_row_style, axis=1)
    .set_table_styles([
        {"selector": "th", "props": [("background-color", "#111827"), ("color", "#F9FAFB"), ("border-color", "#374151")]},
        {"selector": "td", "props": [("font-weight", "500"), ("border-color", "#E5E7EB")]},
    ])
    .hide(axis="index")
)
display(scorecard_style)

print(f"Sample: {df.trade_date.nunique()} trade dates x {df.ts_code.nunique()} stocks"
      f" | valid rows {df.universe.sum():,}"
      f" | factor mom_{FACTOR_WINDOW}d"
      f" | main factor column: {FACTOR_COL} | forward return column: {FWD_COL}")


---
## Step 7. Backtest Review

This is not a production trading backtest. It turns the earlier Q5 bucket into a light portfolio view so we can see whether the signal still stands after simple rebalance costs.

## Simplified rules used here

- Portfolio: long only Q5.
- Weighting: compare equal weight and market-cap weight inside Q5.
- Rebalance: every 5 trading days.
- Return convention: reuse the earlier adjusted close-to-close labels, with the signal formed after the T close and returns measured from T+1 to T+6.
- Costs: fixed two-way rebalance fee, `2 * turnover * one_way_cost`.
- Limit-up filter: if `daily_panel` contains next-open limit-up information, exclude stocks that cannot be bought at next open; otherwise keep the lightweight research setup and do not simulate real fills.


In [ ]:
# Light research backtest: Q5 equal-weight and market-cap-weight portfolios, 5-day rebalance, two-sided cost.
REBALANCE_STEP = 5
ONE_WAY_COST = 0.001
COST_SCENARIOS = [0.0, 0.001, 0.003]
LIMIT_FILTER_COL = "next_open_is_limit_up"


def _market_cap_weighted_return(selected):
    weights = selected["total_mv"] / selected["total_mv"].sum()
    return selected[FWD_COL].mul(weights).sum()


def run_research_backtest(df, quantile=N_QUANTILES, step=REBALANCE_STEP, one_way_cost=ONE_WAY_COST):
    source = df[df.universe & df["quantile"].eq(quantile) & df[FWD_COL].notna()].copy()
    rebalance_dates = sorted(source["trade_date"].unique())[::step]
    has_limit_filter = LIMIT_FILTER_COL in source.columns

    rows = []
    prev_names = set()
    for trade_date in rebalance_dates:
        candidates = source[source["trade_date"].eq(trade_date)]
        if has_limit_filter:
            selected = candidates[~candidates[LIMIT_FILTER_COL].fillna(False)]
        else:
            selected = candidates
        names = set(selected["ts_code"])
        assert names, "Each rebalance date must have at least one tradable Q5 name."

        gross_ew = selected[FWD_COL].mean()
        gross_vw = _market_cap_weighted_return(selected)
        turnover = 1.0 if not prev_names else 1 - len(prev_names & names) / len(names)
        cost = 2 * turnover * one_way_cost
        rows.append({
            "trade_date": trade_date,
            "n_holdings": len(names),
            "limit_up_filtered": len(candidates) - len(selected),
            "gross_return_ew": gross_ew,
            "gross_return_vw": gross_vw,
            "turnover": turnover,
            "cost": cost,
            "net_return_ew": gross_ew - cost,
            "net_return_vw": gross_vw - cost,
        })
        prev_names = names

    result = pd.DataFrame(rows)
    result["gross_equity_ew"] = (1 + result["gross_return_ew"]).cumprod()
    result["gross_equity_vw"] = (1 + result["gross_return_vw"]).cumprod()
    result["net_equity_ew"] = (1 + result["net_return_ew"]).cumprod()
    result["net_equity_vw"] = (1 + result["net_return_vw"]).cumprod()
    result["drawdown_ew"] = result["net_equity_ew"] / result["net_equity_ew"].cummax() - 1
    result["drawdown_vw"] = result["net_equity_vw"] / result["net_equity_vw"].cummax() - 1
    return result


def summarize_backtest(bt, return_col="net_return_ew", equity_col="net_equity_ew", drawdown_col="drawdown_ew", periods_per_year=252 / REBALANCE_STEP):
    ret = bt[return_col]
    equity = bt[equity_col]
    annual_return = equity.iloc[-1] ** (periods_per_year / len(bt)) - 1
    annual_vol = ret.std() * np.sqrt(periods_per_year)
    return pd.DataFrame([
        ["Rebalances", f"{len(bt):,}", "-"],
        ["Avg holdings", f'{bt["n_holdings"].mean():.1f}', "-"],
        ["Avg period return", f"{ret.mean():.4%}", "Pass" if ret.mean() > 0 else "Weak"],
        ["Cumulative net return", f"{equity.iloc[-1] - 1:.4%}", "Pass" if equity.iloc[-1] > 1 else "Weak"],
        ["Annual return", f"{annual_return:.4%}", "-"],
        ["Annual volatility", f"{annual_vol:.4%}", "-"],
        ["Simple Sharpe", f"{annual_return / annual_vol:.3f}" if annual_vol > 0 else "nan", "-"],
        ["Max drawdown", f'{bt[drawdown_col].min():.4%}', "-"],
        ["Avg turnover", f'{bt["turnover"].mean():.1%}', "-"],
        ["Avg two-sided cost", f'{bt["cost"].mean():.4%}', "-"],
        ["Avg limit-up filtered", f'{bt["limit_up_filtered"].mean():.1f}', "-"],
    ], columns=["Metric", "Value", "Judgement"])


bt = run_research_backtest(df)
bt_summary = summarize_backtest(bt)
display(bt_summary)

weight_compare = pd.DataFrame([
    {"weighting": "Equal Weight", "final_equity": bt["net_equity_ew"].iloc[-1],
     "cum_return": bt["net_equity_ew"].iloc[-1] - 1, "max_drawdown": bt["drawdown_ew"].min()},
    {"weighting": "Market-Cap Weight", "final_equity": bt["net_equity_vw"].iloc[-1],
     "cum_return": bt["net_equity_vw"].iloc[-1] - 1, "max_drawdown": bt["drawdown_vw"].min()},
])
display(weight_compare.set_index("weighting"))

cost_rows = []
for cost in COST_SCENARIOS:
    scenario = run_research_backtest(df, one_way_cost=cost)
    cost_rows.append({
        "one_way_cost": cost,
        "ew_cum_return": scenario["net_equity_ew"].iloc[-1] - 1,
        "vw_cum_return": scenario["net_equity_vw"].iloc[-1] - 1,
        "ew_max_drawdown": scenario["drawdown_ew"].min(),
        "vw_max_drawdown": scenario["drawdown_vw"].min(),
    })
cost_sensitivity = pd.DataFrame(cost_rows)
display(cost_sensitivity.set_index("one_way_cost"))

if LIMIT_FILTER_COL in df.columns:
    print("Limit-up filter enabled: removed Q5 names whose next open is at the up-limit price.")
else:
    print("Limit-up filter not enabled: daily_panel lacks open/up_limit-derived next-open fields.")

fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))

ax = axes[0]
ax.plot(bt["trade_date"], bt["gross_equity_ew"], color=COLORS["blue"], linewidth=1.8, label="EW Gross")
ax.plot(bt["trade_date"], bt["net_equity_ew"], color=COLORS["green"], linewidth=2.0, label="EW Net")
ax.plot(bt["trade_date"], bt["net_equity_vw"], color=COLORS["orange"], linewidth=2.0, label="VW Net")
ax.axhline(1.0, color=COLORS["gray"], linestyle="--", linewidth=0.8)
ax.set_title("Q5 Research Backtest Equity")
ax.legend(frameon=True)

ax = axes[1]
ax.fill_between(bt["trade_date"], bt["drawdown_ew"], 0, color=COLORS["red"], alpha=0.18)
ax.plot(bt["trade_date"], bt["drawdown_ew"], color=COLORS["red"], linewidth=1.6, label="EW")
ax.plot(bt["trade_date"], bt["drawdown_vw"], color=COLORS["orange"], linewidth=1.6, label="VW")
ax.set_title("Net Equity Drawdown")
ax.legend(frameon=False)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))

ax = axes[2]
labels = [f"{c:.1%}" for c in cost_sensitivity["one_way_cost"]]
x = np.arange(len(labels))
width = 0.36
bars_ew = ax.bar(x - width / 2, cost_sensitivity["ew_cum_return"], width, color=COLORS["blue"], label="EW")
bars_vw = ax.bar(x + width / 2, cost_sensitivity["vw_cum_return"], width, color=COLORS["orange"], label="VW")
for bars in (bars_ew, bars_vw):
    for bar in bars:
        val = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, val, f"{val:.1%}", ha="center",
                va="bottom" if val >= 0 else "top", fontsize=8, color="#374151")
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.axhline(0, color=COLORS["gray"], linestyle="--", linewidth=0.8)
ax.set_title("Cost Sensitivity")
ax.set_xlabel("One-Way Cost")
ax.legend(frameon=False)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))

fig.suptitle("Research Backtest Analysis", fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()
